In [28]:
import os
import asyncio
from typing import List

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

In [18]:
# --- Configuration ---
# Ensure your GOOGLE_API_KEY environment variable is set.
try:
    # A model with function/tool calling capabilities is required.
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
    print(f"✅ Language model initialized: {llm.model}")
except Exception as e:
    print(f"🛑 Error initializing language model: {e}")
    llm = None

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Language model initialized: gemini-2.5-flash-lite


In [19]:
# --- Define a Tool ---
@tool
def search_information(query: str) -> str:
    """
    Provides factual information on a given topic. Use this tool to find answers to questions
    like 'What is the capital of France?' or 'What is the weather in London?'.
    """
    print(f"\n--- 🛠️ Tool Called: search_information with query: '{query}' ---")
    # Simulate a search tool with a dictionary of predefined results.
    simulated_results = {
        "weather in london": "The weather in London is currently cloudy with a temperature of 15°C.",
        "capital of france": "The capital of France is Paris.",
        "population of earth": "The estimated population of Earth is around 8 billion people.",
        "tallest mountain": "Mount Everest is the tallest mountain above sea level.",
        "default": f"Simulated search result for '{query}': No specific information found, but the topic seems interesting."
    }
    result = simulated_results.get(query.lower(), simulated_results["default"])
    print(f"--- TOOL RESULT: {result} ---")
    return result

tools = [search_information]

In [22]:
if llm:
    # LangGraph completely replaces AgentExecutor and handles the prompt formatting natively
    agent_executor = create_agent(llm, tools)

In [29]:
async def run_agent_with_tool(query: str):
    """Invokes the LangGraph agent and extracts the final response."""
    print(f"\n--- 🏃 Running Agent with Query: '{query}' ---")
    try:
        # We wrap the text query in a HumanMessage object here
        response = await agent_executor.ainvoke(
            {"messages": [HumanMessage(content=query)]}
        )
        print("\n--- ✅ Final Agent Response ---")
        # The final answer is the content of the last message in the state
        print(response["messages"][-1].content)
    except Exception as e:
        print(f"\n🛑 An error occurred during agent execution: {e}")

In [30]:
async def main():
    """Runs all agent queries concurrently."""
    tasks = [
        run_agent_with_tool("What is the capital of France?"),
        run_agent_with_tool("What's the weather like in London?"),
        run_agent_with_tool("Tell me something about dogs.") # Should trigger the default
    ]
    await asyncio.gather(*tasks)

In [31]:
if llm:
    await main()
else:
    print("\nSkipping agent execution due to LLM initialization failure.")


--- 🏃 Running Agent with Query: 'What is the capital of France?' ---

--- 🏃 Running Agent with Query: 'What's the weather like in London?' ---

--- 🏃 Running Agent with Query: 'Tell me something about dogs.' ---

--- 🛠️ Tool Called: search_information with query: 'What is the weather in London?' ---
--- 🛠️ Tool Called: search_information with query: 'dogs' ---
--- TOOL RESULT: Simulated search result for 'dogs': No specific information found, but the topic seems interesting. ---

--- TOOL RESULT: Simulated search result for 'What is the weather in London?': No specific information found, but the topic seems interesting. ---

--- 🛠️ Tool Called: search_information with query: 'What is the capital of France?' ---
--- TOOL RESULT: Simulated search result for 'What is the capital of France?': No specific information found, but the topic seems interesting. ---

--- ✅ Final Agent Response ---
I am sorry, I cannot provide real-time information such as weather conditions.

--- ✅ Final Agent R

In [33]:
import os, getpass
import asyncio
import nest_asyncio
from typing import List
from dotenv import load_dotenv
import logging

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool as langchain_tool
from langchain.agents import create_agent

# UNCOMMENT
# Prompt the user securely and set API keys as an environment variables
# os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API key: ")
# os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

In [34]:

try:
   # A model with function/tool calling capabilities is required.
   llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
   print(f"✅ Language model initialized: {llm.model}")
except Exception as e:
   print(f"🛑 Error initializing language model: {e}")
   llm = None

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Language model initialized: gemini-2.5-flash-lite


In [35]:
# --- Define a Tool ---
@langchain_tool
def search_information(query: str) -> str:
   """
   Provides factual information on a given topic. Use this tool to find answers to phrases
   like 'capital of France' or 'weather in London?'.
   """
   print(f"\n--- 🛠️ Tool Called: search_information with query: '{query}' ---")
   # Simulate a search tool with a dictionary of predefined results.
   simulated_results = {
       "weather in london": "The weather in London is currently cloudy with a temperature of 15°C.",
       "capital of france": "The capital of France is Paris.",
       "population of earth": "The estimated population of Earth is around 8 billion people.",
       "tallest mountain": "Mount Everest is the tallest mountain above sea level.",
       "default": f"Simulated search result for '{query}': No specific information found, but the topic seems interesting."
   }
   result = simulated_results.get(query.lower(), simulated_results["default"])
   print(f"--- TOOL RESULT: {result} ---")
   return result

tools = [search_information]


In [37]:

# --- Create a Tool-Calling Agent ---
if llm:

   # Create the agent, binding the LLM, tools, and prompt together.
   agent_executor = create_agent(llm, tools)

In [39]:
async def run_agent_with_tool(query: str):
    """Invokes the LangGraph agent and extracts the final response."""
    print(f"\n--- 🏃 Running Agent with Query: '{query}' ---")
    try:
        # We wrap the text query in a HumanMessage object here
        response = await agent_executor.ainvoke(
            {"messages": [HumanMessage(content=query)]}
        )
        print("\n--- ✅ Final Agent Response ---")
        # The final answer is the content of the last message in the state
        print(response["messages"][-1].content)
    except Exception as e:
        print(f"\n🛑 An error occurred during agent execution: {e}")

In [40]:

nest_asyncio.apply()
asyncio.run(main())


--- 🏃 Running Agent with Query: 'What is the capital of France?' ---

--- 🏃 Running Agent with Query: 'What's the weather like in London?' ---

--- 🏃 Running Agent with Query: 'Tell me something about dogs.' ---

--- 🛠️ Tool Called: search_information with query: 'weather in London' ---
--- TOOL RESULT: The weather in London is currently cloudy with a temperature of 15°C. ---

--- 🛠️ Tool Called: search_information with query: 'capital of France' ---
--- TOOL RESULT: The capital of France is Paris. ---

--- 🛠️ Tool Called: search_information with query: 'dogs' ---
--- TOOL RESULT: Simulated search result for 'dogs': No specific information found, but the topic seems interesting. ---

--- ✅ Final Agent Response ---
The weather in London is currently cloudy with a temperature of 15°C.

--- ✅ Final Agent Response ---
The capital of France is Paris.

--- ✅ Final Agent Response ---
The search didn't yield specific information, but dogs are fascinating animals! They are domesticated mammals